In [1]:
#Download Well Data for British Columbia (Canada) from https://reports.bc-er.ca/ogc/f?p=200:8:15361179614398::NO:::

In [2]:
# Mount google drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
import pandas as pd

# Assuming your CSV file is named 'your_file.csv'
csv_file = '/content/drive/Shareddrives/GEM Shared Drive/Programs/Oil & Gas Program/Global Oil and Gas Extraction Tracker (GOGET)/Data sets & reports (O&G)/O&G country materials/Canada (GOGET O&G)/British Columbia/BC Total Production 2023.csv'

# Read the CSV file into a pandas DataFrame
df = pd.read_csv(csv_file)

# Convert relevant columns to numeric
numeric_columns = ['Production Days', 'Gas Production (e3m3)', 'Oil Production (m3)', 'Condensate Production (m3)']
df[numeric_columns] = df[numeric_columns].apply(pd.to_numeric, errors='coerce')

In [4]:
# Filter for 2023 production data
data_2023 = df[df['Zone Prod Period'].astype(str).str.startswith('2023')]

data_2023.head()

,Well Authorization Number,Completion Event Sequence,UWI,Zone Prod Period,Area Code,Formtn Code,Well Area Name,Oper Abbreviation,Production Days,Gas Production (e3m3),Oil Production (m3),Condensate Production (m3),Water Production (m3),Marketable Gas Volume (e3m3),Ethane Sales Volume (m3),Propane Sales Volume (m3),Butane Sales Volume (m3),Pentane Sales Volume (m3)
0,3035,0,100111908513W600,202312,2000,4535,BOUNDARY,WHITECAP,31.0,6.9,70.0,0.0,555.5,NaN,NaN,NaN,NaN,NaN
1,2520,0,100071208614W600,202310,2000,4535,BOUNDARY,WHITECAP,31.0,3.4,16.0,0.0,668.4,NaN,NaN,NaN,NaN,NaN
2,2375,0,100070108614W600,202312,2000,4535,BOUNDARY,WHITECAP,31.0,4.3,8.6,0.0,244.1,NaN,NaN,NaN,NaN,NaN
3,27707,0,202A056H094I0900,202302,4650,2600,HAY,HARVEST,28.0,10.0,192.7,0.0,7084.1,NaN,NaN,NaN,NaN,NaN
4,17011,0,203D021G094I0900,202302,4650,2600,HAY,HARVEST,28.0,8.7,59.9,0.0,1099.7,NaN,NaN,NaN,NaN,NaN


In [6]:
# Group by Well Area Name and Operator Abbreviation and sum gas, oil, and condensate
totals_2023 = data_2023.groupby(['Well Area Name', 'Oper Abbreviation']).agg({
    'Gas Production (e3m3)': 'sum',
    'Oil Production (m3)': 'sum',
    'Condensate Production (m3)': 'sum',
    'UWI': lambda x: ';'.join(sorted(x.unique()))
}).reset_index()

totals_2023.head()

,Well Area Name,Oper Abbreviation,Gas Production (e3m3),Oil Production (m3),Condensate Production (m3),UWI
0,ALTARES,PACIFIC CANBRIAM,14884.8,0.0,0.0,102010608525W602
1,BRASSEY,CVE ENERGY,775.6,0.0,6.7,100010407719W600
2,ELLEH,NTE CANADA,335.7,0.0,0.9,200D021D094I1200
3,GROUNDBIRCH,SHELL,2962.3,0.0,0.0,102110908020W600
4,GUNNELL,NTE CANADA,1713.3,0.0,21.4,200B024B094P0400;200B087K094I1200;200D067K094I...


In [14]:
# Conversion factors
gas_to_boe_factor = 5.8 / 1000  # 1 e3m³ of gas equals 5.8 BOE, so in million BOE, it's 0.0058 MMboe
m3_to_bbl_factor = 6.2898  # 1 m³ of oil or condensate equals 6.2898 barrels (bbl)

# Convert to MMBOE and bbl
totals_2023['Gas (MMBOE)'] = totals_2023['Gas Production (e3m3)'] * gas_to_boe_factor
totals_2023['Oil (bbl)'] = totals_2023['Oil Production (m3)'] * m3_to_bbl_factor
totals_2023['Condensate (bbl)'] = totals_2023['Condensate Production (m3)'] * m3_to_bbl_factor

# Total Hydrocarbons in BOE
totals_2023['Total Hydrocarbons (BOE)'] = totals_2023['Gas (BOE)'] + totals_2023['Oil (bbl)'] + totals_2023['Condensate (bbl)']
totals_2023['Total Hydrocarbons (MMBOE)'] = totals_2023['Total Hydrocarbons (BOE)'] / 1e6

totals_2023.head()

,Well Area Name,Oper Abbreviation,Gas Production (e3m3),Oil Production (m3),Condensate Production (m3),UWI,Gas (BOE),Oil (bbl),Condensate (bbl),Total Hydrocarbons (BOE),Total Hydrocarbons (MMBOE),Gas (MMBOE)
0,ALTARES,PACIFIC CANBRIAM,14884.8,0.0,0.0,102010608525W602,89.3088,0.0,0.00000,89.30880,0.000089,86.33184
1,BRASSEY,CVE ENERGY,775.6,0.0,6.7,100010407719W600,4.6536,0.0,42.14166,46.79526,0.000047,4.49848
2,ELLEH,NTE CANADA,335.7,0.0,0.9,200D021D094I1200,2.0142,0.0,5.66082,7.67502,0.000008,1.94706
3,GROUNDBIRCH,SHELL,2962.3,0.0,0.0,102110908020W600,17.7738,0.0,0.00000,17.77380,0.000018,17.18134
4,GUNNELL,NTE CANADA,1713.3,0.0,21.4,200B024B094P0400;200B087K094I1200;200D067K094I...,10.2798,0.0,134.60172,144.88152,0.000145,9.93714


In [15]:
# Sort by Total Hydrocarbons (MMBOE), high to low
totals_2023_sorted = totals_2023.sort_values(by='Total Hydrocarbons (MMBOE)', ascending=False)

totals_2023_sorted.head()

,Well Area Name,Oper Abbreviation,Gas Production (e3m3),Oil Production (m3),Condensate Production (m3),UWI,Gas (BOE),Oil (bbl),Condensate (bbl),Total Hydrocarbons (BOE),Total Hydrocarbons (MMBOE),Gas (MMBOE)
310,SUNRISE,OVV,10500465.7,0.0,342065.7,100010508017W600;100010607916W600;100010707916...,63002.7942,0.00000,2.151525e+06,2.214528e+06,2.214528,60902.70106
336,TOWERLAKE,OVV,921878.4,0.0,297211.3,100020408117W600;100023108017W600;100030408117...,5531.2704,0.00000,1.869400e+06,1.874931e+06,1.874931,5346.89472
338,TOWN,PETRONAS,3636317.4,0.0,140908.3,200A005H094B1600;200A007B094G0100;200A008F094G...,21817.9044,0.00000,8.862850e+05,9.081029e+05,0.908103,21090.64092
49,BOUNDARY,WHITECAP,20352.2,130989.9,34.3,100020808613W600;100021308414W600;100021908413...,122.1132,823900.27302,2.157401e+02,8.242381e+05,0.824238,118.04276
119,FIREWEED,CNRL,378404.9,6596.3,123091.7,200A011G094A1302;200A012G094A1300;200A016H094A...,2270.4294,41489.40774,7.742222e+05,8.179820e+05,0.817982,2194.74842


In [16]:
# Print the rows that contribute to duplicates
duplicates = totals_2023_sorted[totals_2023_sorted.duplicated(subset=['Well Area Name', 'Oper Abbreviation'], keep=False)]
print("Rows contributing to duplicates:")
print(duplicates)

Rows contributing to duplicates:
Empty DataFrame
Columns: [Well Area Name, Oper Abbreviation, Gas Production (e3m3), Oil Production (m3), Condensate Production (m3), UWI, Gas (BOE), Oil (bbl), Condensate (bbl), Total Hydrocarbons (BOE), Total Hydrocarbons (MMBOE), Gas (MMBOE)]
Index: []


In [17]:
# save to excel format
totals_2023_sorted.to_excel('/content/drive/Shareddrives/GEM Shared Drive/Programs/Oil & Gas Program/Global Oil and Gas Extraction Tracker (GOGET)/Data sets & reports (O&G)/O&G country materials/Canada (GOGET O&G)/British Columbia/BC_2023_production2.xlsx', index=False)